In [1]:
%pip install pyprind
%pip install pandas
%pip install numpy
%pip install scikit-learn
%pip install curl-cffi
%pip install torch

  Using cached PyPrind-2.11.3-py2.py3-none-any.whl.metadata (1.1 kB)
Using cached PyPrind-2.11.3-py2.py3-none-any.whl (8.4 kB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 53.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 63.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp

## Downloading the Datasets

In [2]:
from curl_cffi import requests

url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
# Use 'impersonate' to mimic a real browser (e.g., chrome124)
response = requests.get(url, impersonate="chrome124")

# Check if the request was successful
if response.status_code == 200:
    with open("aclImdb_v1.tar.gz", "wb") as f:
        f.write(response.content)
    print("File downloaded successfully.")

File downloaded successfully.


In [3]:
import tarfile
with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
    tar.extractall()

/var/folders/84/tslxp8q17534xdq_c084fw5w0000gn/T/ipykernel_19495/219513043.py:3: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


## Preprocessing the Dataset into a more convenient format

In [4]:
import pyprind
import pandas as pd
import os
import sys

basepath = 'aclImdb'
labels = {'pos': 1, 'neg': 0}
pbar = pyprind.ProgBar(50000, stream=sys.stdout)

data = []

for s in ('test', 'train'):
    for l in ('pos', 'neg'):
        path = os.path.join(basepath, s, l)
        for file in sorted(os.listdir(path)):
            with open(os.path.join(path, file), 'r', encoding='utf-8') as infile:
                txt = infile.read()
            
            data.append([txt, labels[l]]) 
            pbar.update()

df = pd.DataFrame(data, columns=['review', 'sentiment'])


0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:02


## Store the datasets as CSV file

In [5]:
import numpy as np
np.random.seed(0)
df=df.reindex(np.random.permutation(df.index))
df.to_csv('movie_data.csv', index=False, encoding='utf-8')
df = pd.read_csv('movie_data.csv', encoding='utf-8')
df = df.rename(columns={"0": "review", "1": "sentiments"})
df.head(3)

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0


## Transform to tf-idf

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

def get_tfidf_features(X_train, X_test):
    tfidf = TfidfVectorizer(use_idf=True, norm='l2', smooth_idf=True,
                            strip_accents=None, lowercase=False, preprocessor=None)
    
    X_train_tfidf = tfidf.fit_transform(X_train)  # fit only on train
    X_test_tfidf  = tfidf.transform(X_test)        # transform using same vocab
    
    return X_train_tfidf, X_test_tfidf, tfidf

## Cleaning Text Data

In [7]:
import re
def preprocessor(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(r'(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub(r'[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    
    return text
    
df['review'] = df['review'].apply(preprocessor)

## Train/Test Split Data

In [8]:
from sklearn.model_selection import train_test_split

X = df['review'].values
y = df['sentiment'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

In [9]:
X_train_tfidf, X_test_tfidf, tfidf = get_tfidf_features(X_train, X_test)
print(X_train_tfidf.shape)  
print(X_test_tfidf.shape)   

(35000, 89509)
(15000, 89509)


## Building an FNN Classifier

In [10]:
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.sparse import issparse
import torch.nn as nn
import numpy as np

class SparseDataset(Dataset):
    def __init__(self, X, y):
        self.X = X  # stays as sparse matrix
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx].toarray().squeeze(), dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y

train_dataset = SparseDataset(X_train_tfidf, y_train)
test_dataset  = SparseDataset(X_test_tfidf,  y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

input_size = X_train_tfidf.shape[1]
print(f"Input size (vocab size): {input_size}")

Input size (vocab size): 89509


## Define the FNN

In [11]:
class FNN_Large(nn.Module):
    def __init__(self, input_size):
        super(FNN_Large, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

model = FNN_Large(input_size)
print(model)

FNN_Large(
  (network): Sequential(
    (0): Linear(in_features=89509, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


## Training a Baseline Model and Tuning Hyperparameters

In [12]:
import time

def train_model(model, train_loader, test_loader, lr=0.001, weight_decay=1e-4, epochs=10):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch).squeeze()
            loss  = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            train_correct, train_total = 0, 0
            for X_batch, y_batch in train_loader:
                preds = model(X_batch).squeeze()
                train_correct += ((preds >= 0.5) == y_batch).sum().item()
                train_total   += y_batch.size(0)

            test_correct, test_total = 0, 0
            for X_batch, y_batch in test_loader:
                preds = model(X_batch).squeeze()
                test_correct += ((preds >= 0.5) == y_batch).sum().item()
                test_total   += y_batch.size(0)

        train_acc = train_correct / train_total
        test_acc  = test_correct  / test_total
        print(f"Epoch {epoch+1}/{epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    
    elapsed = time.time() - start
    print(f"\nTime: {elapsed:.2f}s | Final Test Accuracy: {test_acc:.4f}")
    return test_acc, elapsed

In [13]:
print("=" * 50)
print("Baseline FNN: 3 hidden layers [256 → 128 → 64]")
print("=" * 50)
baseline_model = FNN_Large(input_size)
baseline_acc, baseline_time = train_model(baseline_model, train_loader, test_loader)

Baseline FNN: 3 hidden layers [256 → 128 → 64]
Epoch 1/10 | Train Acc: 0.9595 | Test Acc: 0.8984
Epoch 2/10 | Train Acc: 0.9864 | Test Acc: 0.8930
Epoch 3/10 | Train Acc: 0.9875 | Test Acc: 0.8841
Epoch 4/10 | Train Acc: 0.9880 | Test Acc: 0.8793
Epoch 5/10 | Train Acc: 0.9897 | Test Acc: 0.8778
Epoch 6/10 | Train Acc: 0.9925 | Test Acc: 0.8739
Epoch 7/10 | Train Acc: 0.9914 | Test Acc: 0.8760
Epoch 8/10 | Train Acc: 0.9931 | Test Acc: 0.8753
Epoch 9/10 | Train Acc: 0.9906 | Test Acc: 0.8727
Epoch 10/10 | Train Acc: 0.9944 | Test Acc: 0.8749

Time: 179.70s | Final Test Accuracy: 0.8749


In [14]:
results = []

configs = [
    {'lr': 0.001,  'weight_decay': 1e-4},   # default
    {'lr': 0.0005, 'weight_decay': 1e-5},   # lower lr
    {'lr': 0.01,   'weight_decay': 1e-3},   # higher lr
]

for cfg in configs:
    print(f"\n--- lr={cfg['lr']}, weight_decay={cfg['weight_decay']} ---")
    model = FNN_Large(input_size)
    acc, t = train_model(model, train_loader, test_loader,
                         lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    results.append({'lr': cfg['lr'], 'weight_decay': cfg['weight_decay'],
                    'test_acc': acc, 'time': t})


--- lr=0.001, weight_decay=0.0001 ---
Epoch 1/10 | Train Acc: 0.9641 | Test Acc: 0.9019
Epoch 2/10 | Train Acc: 0.9866 | Test Acc: 0.8933
Epoch 3/10 | Train Acc: 0.9885 | Test Acc: 0.8835
Epoch 4/10 | Train Acc: 0.9862 | Test Acc: 0.8787
Epoch 5/10 | Train Acc: 0.9890 | Test Acc: 0.8803
Epoch 6/10 | Train Acc: 0.9922 | Test Acc: 0.8764
Epoch 7/10 | Train Acc: 0.9951 | Test Acc: 0.8781
Epoch 8/10 | Train Acc: 0.9895 | Test Acc: 0.8741
Epoch 9/10 | Train Acc: 0.9946 | Test Acc: 0.8771
Epoch 10/10 | Train Acc: 0.9934 | Test Acc: 0.8733

Time: 180.63s | Final Test Accuracy: 0.8733

--- lr=0.0005, weight_decay=1e-05 ---
Epoch 1/10 | Train Acc: 0.9550 | Test Acc: 0.9072
Epoch 2/10 | Train Acc: 0.9878 | Test Acc: 0.8995
Epoch 3/10 | Train Acc: 0.9973 | Test Acc: 0.8941
Epoch 4/10 | Train Acc: 0.9991 | Test Acc: 0.8902
Epoch 5/10 | Train Acc: 0.9997 | Test Acc: 0.8887
Epoch 6/10 | Train Acc: 0.9999 | Test Acc: 0.8880
Epoch 7/10 | Train Acc: 1.0000 | Test Acc: 0.8882
Epoch 8/10 | Train Acc: 1.

In [15]:
print("\n" + "=" * 60)
print("Comparison: FNN vs Logistic Regression (Book)")
print("=" * 60)
print(f"{'Model':<35} {'Test Acc':>10} {'Time':>10}")
print("-" * 60)
print(f"{'Logistic Regression (book)':<35} {'0.899':>10} {'5-10 min':>10}")
print(f"{'FNN Baseline [128→64]':<35} {baseline_acc:>10.3f} {baseline_time:>9.1f}s")
for r in results:
    label = f"FNN Large [256→128→64] lr={r['lr']}"
    print(f"{label:<35} {r['test_acc']:>10.3f} {r['time']:>9.1f}s")


Comparison: FNN vs Logistic Regression (Book)
Model                                 Test Acc       Time
------------------------------------------------------------
Logistic Regression (book)               0.899   5-10 min
FNN Baseline [128→64]                    0.875     179.7s
FNN Large [256→128→64] lr=0.001          0.873     180.6s
FNN Large [256→128→64] lr=0.0005         0.890     180.6s
FNN Large [256→128→64] lr=0.01           0.886     197.6s


## K-fold Cross Validation with PyTorch

In [18]:
import torch
from torch import nn
from torch.utils.data import ConcatDataset
from sklearn.model_selection import KFold
import time

def reset_weights(m):
    for layer in m.children():
        if hasattr(layer, 'reset_parameters'):
            layer.reset_parameters()

def kfold_cross_validation(k_folds=5, num_epochs=10, lr=0.0005, weight_decay=1e-5, batch_size=256):
    criterion = nn.BCELoss()
    torch.manual_seed(42)

    dataset = ConcatDataset([train_dataset, test_dataset])
    kfold = KFold(n_splits=k_folds, shuffle=True, random_state=42)

    fold_results = []
    total_start = time.time()

    print(f'K-FOLD CROSS VALIDATION (k={k_folds}, epochs={num_epochs}, lr={lr})')
    print('=' * 60)

    for fold, (train_ids, test_ids) in enumerate(kfold.split(dataset)):
        fold_start = time.time()
        print(f'\nFOLD {fold + 1}/{k_folds}')
        print('-' * 40)

        train_subsampler = torch.utils.data.SubsetRandomSampler(train_ids)
        test_subsampler = torch.utils.data.SubsetRandomSampler(test_ids)

        trainloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=train_subsampler)
        testloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, sampler=test_subsampler)

        network = FNN_Large(input_size)
        network.apply(reset_weights)
        optimizer = torch.optim.Adam(network.parameters(), lr=lr, weight_decay=weight_decay)

        for epoch in range(num_epochs):
            network.train()
            for inputs, targets in trainloader:
                optimizer.zero_grad()
                outputs = network(inputs).squeeze()
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

        network.eval()
        with torch.no_grad():
            train_correct, train_total = 0, 0
            for inputs, targets in trainloader:
                preds = network(inputs).squeeze()
                train_correct += ((preds >= 0.5).float() == targets).sum().item()
                train_total += targets.size(0)

            test_correct, test_total = 0, 0
            for inputs, targets in testloader:
                preds = network(inputs).squeeze()
                test_correct += ((preds >= 0.5).float() == targets).sum().item()
                test_total += targets.size(0)

        fold_time = time.time() - fold_start
        train_acc = train_correct / train_total
        test_acc = test_correct / test_total

        print(f'  Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Time: {fold_time:.1f}s')
        fold_results.append({
            'fold': fold + 1,
            'train_acc': train_acc,
            'test_acc': test_acc,
            'time': fold_time
        })

    total_time = time.time() - total_start

    avg_train = sum(r['train_acc'] for r in fold_results) / len(fold_results)
    avg_test = sum(r['test_acc'] for r in fold_results) / len(fold_results)

    print('\n' + '=' * 60)
    print(f'RESULTS FOR {k_folds}-FOLD CROSS VALIDATION')
    print('=' * 60)
    print(f"{'Fold':<8} {'Train Acc':>12} {'Test Acc':>12} {'Time':>10}")
    print('-' * 45)
    for r in fold_results:
        print(f"Fold {r['fold']:<4} {r['train_acc']:>12.4f} {r['test_acc']:>12.4f} {r['time']:>9.1f}s")
    print('-' * 45)
    print(f"{'Avg':<8} {avg_train:>12.4f} {avg_test:>12.4f} {total_time:>9.1f}s")

    return fold_results, avg_train, avg_test, total_time

kfold_results_5, avg_train_5, avg_test_5, kfold_time_5 = kfold_cross_validation(k_folds=5)

K-FOLD CROSS VALIDATION (k=5, epochs=10, lr=0.0005)

FOLD 1/5
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8857 | Time: 137.7s

FOLD 2/5
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8857 | Time: 145.5s

FOLD 3/5
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8834 | Time: 156.6s

FOLD 4/5
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8837 | Time: 174.3s

FOLD 5/5
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8905 | Time: 192.2s

RESULTS FOR 5-FOLD CROSS VALIDATION
Fold        Train Acc     Test Acc       Time
---------------------------------------------
Fold 1          1.0000       0.8857     137.7s
Fold 2          1.0000       0.8857     145.5s
Fold 3          1.0000       0.8834     156.6s
Fold 4          1.0000       0.8837     174.3s
Fold 5          1.0000       0.8905     192.2s
---------------------------------------------
Avg          

In [19]:
print("Tuning k: trying k = 3 and k = 10")
print()

kfold_results_3, avg_train_3, avg_test_3, kfold_time_3 = kfold_cross_validation(k_folds=3)
print()
kfold_results_10, avg_train_10, avg_test_10, kfold_time_10 = kfold_cross_validation(k_folds=10)

Tuning k: trying k = 3 and k = 10

K-FOLD CROSS VALIDATION (k=3, epochs=10, lr=0.0005)

FOLD 1/3
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8894 | Time: 113.8s

FOLD 2/3
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8828 | Time: 123.5s

FOLD 3/3
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8880 | Time: 140.1s

RESULTS FOR 3-FOLD CROSS VALIDATION
Fold        Train Acc     Test Acc       Time
---------------------------------------------
Fold 1          1.0000       0.8894     113.8s
Fold 2          1.0000       0.8828     123.5s
Fold 3          1.0000       0.8880     140.1s
---------------------------------------------
Avg            1.0000       0.8867     377.4s

K-FOLD CROSS VALIDATION (k=10, epochs=10, lr=0.0005)

FOLD 1/10
----------------------------------------
  Train Acc: 0.9817 | Test Acc: 0.8810 | Time: 196.4s

FOLD 2/10
----------------------------------------
  Train Acc: 1.0000 | Te

In [20]:
print("=" * 65)
print("COMPARISON: K-Fold Cross Validation vs Baseline (no K-Fold)")
print("=" * 65)
print(f"{'Model':<35} {'Train Acc':>10} {'Test Acc':>10} {'Time':>10}")
print("-" * 65)
print(f"{'Baseline (no K-Fold)':<35} {'N/A':>10} {baseline_acc:>10.4f} {baseline_time:>9.1f}s")
print(f"{'K-Fold (k=3)':<35} {avg_train_3:>10.4f} {avg_test_3:>10.4f} {kfold_time_3:>9.1f}s")
print(f"{'K-Fold (k=5)':<35} {avg_train_5:>10.4f} {avg_test_5:>10.4f} {kfold_time_5:>9.1f}s")
print(f"{'K-Fold (k=10)':<35} {avg_train_10:>10.4f} {avg_test_10:>10.4f} {kfold_time_10:>9.1f}s")
print("-" * 65)

COMPARISON: K-Fold Cross Validation vs Baseline (no K-Fold)
Model                                Train Acc   Test Acc       Time
-----------------------------------------------------------------
Baseline (no K-Fold)                       N/A     0.8749     179.7s
K-Fold (k=3)                            1.0000     0.8867     377.4s
K-Fold (k=5)                            1.0000     0.8858     806.3s
K-Fold (k=10)                           0.9982     0.8838    1744.3s
-----------------------------------------------------------------


In [ ]:
## Task 5: Training using Dropout Regularization

In [21]:
class FNN_Baseline(nn.Module):
    """Baseline FNN from Task 2 (no regularization)."""
    def __init__(self, input_size):
        super(FNN_Baseline, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

class FNN_Dropout(nn.Module):
    """FNN with nn.Dropout layers for regularization (same architecture as baseline)."""
    def __init__(self, input_size, p1=0.5, p2=0.3):
        super(FNN_Dropout, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(p=p1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=p2),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

def train_and_evaluate(model, train_loader, test_loader, lr=0.0005, weight_decay=0, epochs=10, verbose=True):
    """Train a model and return train_acc, test_acc, time, and the trained model."""
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    start = time.time()

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            preds = model(X_batch).squeeze()
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()

        if verbose:
            model.eval()
            with torch.no_grad():
                tc, tt = 0, 0
                for X_batch, y_batch in train_loader:
                    preds = model(X_batch).squeeze()
                    tc += ((preds >= 0.5) == y_batch).sum().item()
                    tt += y_batch.size(0)
                vc, vt = 0, 0
                for X_batch, y_batch in test_loader:
                    preds = model(X_batch).squeeze()
                    vc += ((preds >= 0.5) == y_batch).sum().item()
                    vt += y_batch.size(0)
                print(f"  Epoch {epoch+1}/{epochs} | Train Acc: {tc/tt:.4f} | Test Acc: {vc/vt:.4f}")

    model.eval()
    with torch.no_grad():
        train_correct, train_total = 0, 0
        for X_batch, y_batch in train_loader:
            preds = model(X_batch).squeeze()
            train_correct += ((preds >= 0.5) == y_batch).sum().item()
            train_total += y_batch.size(0)

        test_correct, test_total = 0, 0
        for X_batch, y_batch in test_loader:
            preds = model(X_batch).squeeze()
            test_correct += ((preds >= 0.5) == y_batch).sum().item()
            test_total += y_batch.size(0)

    elapsed = time.time() - start
    train_acc = train_correct / train_total
    test_acc = test_correct / test_total
    return train_acc, test_acc, elapsed, model

print("Training Baseline FNN (no dropout, no regularization) [128 -> 64]")
print("=" * 60)
baseline_no_reg = FNN_Baseline(input_size)
baseline_no_reg_train_acc, baseline_no_reg_acc, baseline_no_reg_time, _ = train_and_evaluate(
    baseline_no_reg, train_loader, test_loader, lr=0.0005, weight_decay=0, epochs=10
)
print(f"\nFinal => Train Acc: {baseline_no_reg_train_acc:.4f} | Test Acc: {baseline_no_reg_acc:.4f} | Time: {baseline_no_reg_time:.1f}s")

Training Baseline FNN (no dropout, no regularization) [128 -> 64]
  Epoch 1/10 | Train Acc: 0.9308 | Test Acc: 0.8958
  Epoch 2/10 | Train Acc: 0.9754 | Test Acc: 0.9087
  Epoch 3/10 | Train Acc: 0.9905 | Test Acc: 0.9058
  Epoch 4/10 | Train Acc: 0.9969 | Test Acc: 0.9015
  Epoch 5/10 | Train Acc: 0.9989 | Test Acc: 0.8978
  Epoch 6/10 | Train Acc: 0.9995 | Test Acc: 0.8941
  Epoch 7/10 | Train Acc: 0.9997 | Test Acc: 0.8917
  Epoch 8/10 | Train Acc: 0.9999 | Test Acc: 0.8907
  Epoch 9/10 | Train Acc: 1.0000 | Test Acc: 0.8897
  Epoch 10/10 | Train Acc: 1.0000 | Test Acc: 0.8890

Final => Train Acc: 1.0000 | Test Acc: 0.8890 | Time: 150.0s


### Task 5.1: Single Dropout Model

In [22]:
dropout_configs = [
    {'p1': 0.3, 'p2': 0.2},
    {'p1': 0.5, 'p2': 0.3},
    {'p1': 0.5, 'p2': 0.5},
]

best_dropout_acc = 0
best_dropout_cfg = None

print("Task 5.1: Tuning a Single Dropout Model")
print("=" * 60)

for cfg in dropout_configs:
    print(f"\nDropout p1={cfg['p1']}, p2={cfg['p2']}")
    print("-" * 40)
    model = FNN_Dropout(input_size, p1=cfg['p1'], p2=cfg['p2'])
    train_acc, test_acc, elapsed, _ = train_and_evaluate(model, train_loader, test_loader)
    print(f"  Final => Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Time: {elapsed:.1f}s")

    if test_acc > best_dropout_acc:
        best_dropout_acc = test_acc
        best_dropout_train_acc = train_acc
        best_dropout_time = elapsed
        best_dropout_cfg = cfg

print(f"\nBest single dropout config: p1={best_dropout_cfg['p1']}, p2={best_dropout_cfg['p2']}")
print(f"  Train Acc: {best_dropout_train_acc:.4f} | Test Acc: {best_dropout_acc:.4f} | Time: {best_dropout_time:.1f}s")

Task 5.1: Tuning a Single Dropout Model

Dropout p1=0.3, p2=0.2
----------------------------------------
  Epoch 1/10 | Train Acc: 0.9232 | Test Acc: 0.8956
  Epoch 2/10 | Train Acc: 0.9664 | Test Acc: 0.9096
  Epoch 3/10 | Train Acc: 0.9854 | Test Acc: 0.9081
  Epoch 4/10 | Train Acc: 0.9937 | Test Acc: 0.9053
  Epoch 5/10 | Train Acc: 0.9975 | Test Acc: 0.9004
  Epoch 6/10 | Train Acc: 0.9989 | Test Acc: 0.8966
  Epoch 7/10 | Train Acc: 0.9995 | Test Acc: 0.8939
  Epoch 8/10 | Train Acc: 0.9997 | Test Acc: 0.8930
  Epoch 9/10 | Train Acc: 0.9999 | Test Acc: 0.8931
  Epoch 10/10 | Train Acc: 0.9999 | Test Acc: 0.8927
  Final => Train Acc: 0.9999 | Test Acc: 0.8927 | Time: 150.3s

Dropout p1=0.5, p2=0.3
----------------------------------------
  Epoch 1/10 | Train Acc: 0.9175 | Test Acc: 0.8925
  Epoch 2/10 | Train Acc: 0.9600 | Test Acc: 0.9094
  Epoch 3/10 | Train Acc: 0.9784 | Test Acc: 0.9073
  Epoch 4/10 | Train Acc: 0.9899 | Test Acc: 0.9061
  Epoch 5/10 | Train Acc: 0.9954 | Tes

### Task 5.2: Bagging Ensemble with Dropout Models (≥ 5 models)

In [23]:
from torch.utils.data import DataLoader, SubsetRandomSampler

ensemble_configs = [
    {'p1': 0.3, 'p2': 0.2},
    {'p1': 0.4, 'p2': 0.2},
    {'p1': 0.5, 'p2': 0.3},
    {'p1': 0.5, 'p2': 0.5},
    {'p1': 0.6, 'p2': 0.4},
]

n_models = len(ensemble_configs)
n_train = len(train_dataset)

ensemble_models = []
ensemble_start = time.time()

print(f"Task 5.2: Bagging Ensemble with {n_models} Dropout Models")
print("=" * 60)

for i, cfg in enumerate(ensemble_configs):
    print(f"\nModel {i+1}/{n_models}: Dropout p1={cfg['p1']}, p2={cfg['p2']}")
    print("-" * 40)

    np.random.seed(i)
    bag_indices = np.random.choice(n_train, size=n_train, replace=True)
    bag_sampler = SubsetRandomSampler(bag_indices)
    bag_loader = DataLoader(train_dataset, batch_size=256, sampler=bag_sampler)

    model = FNN_Dropout(input_size, p1=cfg['p1'], p2=cfg['p2'])
    train_acc, test_acc, elapsed, trained_model = train_and_evaluate(
        model, bag_loader, test_loader, verbose=False
    )
    ensemble_models.append(trained_model)
    print(f"  Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Time: {elapsed:.1f}s")

print("\nEvaluating Bagging Ensemble...")
print("-" * 40)

def evaluate_ensemble(models, loader):
    all_preds = []
    all_targets = []
    for X_batch, y_batch in loader:
        batch_probs = torch.zeros(X_batch.size(0))
        for m in models:
            m.eval()
            with torch.no_grad():
                batch_probs += m(X_batch).squeeze()
        batch_probs /= len(models)
        all_preds.append((batch_probs >= 0.5).float())
        all_targets.append(y_batch)
    preds = torch.cat(all_preds)
    targets = torch.cat(all_targets)
    return (preds == targets).sum().item() / targets.size(0)

ensemble_train_acc = evaluate_ensemble(ensemble_models, train_loader)
ensemble_test_acc = evaluate_ensemble(ensemble_models, test_loader)
ensemble_time = time.time() - ensemble_start

print(f"Ensemble Train Acc: {ensemble_train_acc:.4f}")
print(f"Ensemble Test Acc:  {ensemble_test_acc:.4f}")
print(f"Total Time:         {ensemble_time:.1f}s")

Task 5.2: Bagging Ensemble with 5 Dropout Models

Model 1/5: Dropout p1=0.3, p2=0.2
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8829 | Time: 83.1s

Model 2/5: Dropout p1=0.4, p2=0.2
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8877 | Time: 82.7s

Model 3/5: Dropout p1=0.5, p2=0.3
----------------------------------------
  Train Acc: 1.0000 | Test Acc: 0.8875 | Time: 82.4s

Model 4/5: Dropout p1=0.5, p2=0.5
----------------------------------------
  Train Acc: 0.9999 | Test Acc: 0.8869 | Time: 83.4s

Model 5/5: Dropout p1=0.6, p2=0.4
----------------------------------------
  Train Acc: 0.9998 | Test Acc: 0.8882 | Time: 83.0s

Evaluating Bagging Ensemble...
----------------------------------------
Ensemble Train Acc: 0.9833
Ensemble Test Acc:  0.9001
Total Time:         426.6s


### Task 5: Comparison Summary

In [24]:
print("=" * 70)
print("TASK 5 COMPARISON: Dropout Regularization vs Baseline")
print("=" * 70)
print(f"{'Model':<40} {'Train Acc':>10} {'Test Acc':>10} {'Time':>10}")
print("-" * 70)
print(f"{'Baseline FNN (no regularization)':<40} {baseline_no_reg_train_acc:>10.4f} {baseline_no_reg_acc:>10.4f} {baseline_no_reg_time:>9.1f}s")
print(f"{'Single Dropout (best config)':<40} {best_dropout_train_acc:>10.4f} {best_dropout_acc:>10.4f} {best_dropout_time:>9.1f}s")
print(f"{'Bagging Ensemble (5 dropout models)':<40} {ensemble_train_acc:>10.4f} {ensemble_test_acc:>10.4f} {ensemble_time:>9.1f}s")
print("-" * 70)
print(f"\nBaseline FNN: [128 -> 64], lr=0.0005, no weight decay, no dropout")
print(f"Best single dropout: p1={best_dropout_cfg['p1']}, p2={best_dropout_cfg['p2']}")
print(f"Ensemble configs: {ensemble_configs}")

TASK 5 COMPARISON: Dropout Regularization vs Baseline
Model                                     Train Acc   Test Acc       Time
----------------------------------------------------------------------
Baseline FNN (no regularization)             1.0000     0.8890     150.0s
Single Dropout (best config)                 0.9997     0.8947     147.1s
Bagging Ensemble (5 dropout models)          0.9833     0.9001     426.6s
----------------------------------------------------------------------

Baseline FNN: [128 -> 64], lr=0.0005, no weight decay, no dropout
Best single dropout: p1=0.5, p2=0.5
Ensemble configs: [{'p1': 0.3, 'p2': 0.2}, {'p1': 0.4, 'p2': 0.2}, {'p1': 0.5, 'p2': 0.3}, {'p1': 0.5, 'p2': 0.5}, {'p1': 0.6, 'p2': 0.4}]
